# Simulating Xenium-like data (cell-resolution)

This notebook reproduces the **exact configuration** used to validate
`albis`'s Xenium-like, single-cell-resolution output against real Xenium
data in the manuscript's Figure 2 (the `cell_vs_non_diseased_lung` /
`cell_vs_lung_cancer` count-distribution comparisons).

It uses the same low-level `simulate_3d_molecule_sphere_multires` API as
`lower_level_api_tutorial.ipynb` -- the higher-level `generate_data()`
wrapper does not expose `theta`/`theta_jitter`/`base_gene_lognormal`/
`batch_sigma`, all of which this configuration tunes away from their
defaults.

## Setup

Install `albis` with the notebook + plotting extras, either from a clone of
this repo or from PyPI once published (same two options as the package
`README.md`):

```bash
pip install -e ".[tutorial,plot]"      # from a clone
# or: pip install "albis[tutorial,plot]"
```

Then launch Jupyter:

```bash
jupyter-notebook --no-browser --ip=0.0.0.0 --port 8888
```


In [ ]:
from pathlib import Path
import sys

def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "albis" / "__init__.py").is_file():
            return candidate
        nested = candidate / "albis"
        if (nested / "pyproject.toml").is_file() and (nested / "albis" / "__init__.py").is_file():
            return nested
    raise RuntimeError("Could not find the albis repository root.")

repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

sys.modules.pop("albis", None)
import albis as ab
import numpy as np

print("Python:", sys.executable)
print("albis module:", getattr(ab, "__file__", "<no __file__>"))
print("albis version:", getattr(ab, "__version__", "<no __version__>"))


## The exact Figure 2 "cell" configuration

Grouped the same way as `lower_level_api_tutorial.ipynb`'s config dict.
Comments mark which values are manuscript-specific tuning vs. left at the
simulator's own defaults. The two knobs that make this "Xenium-like"
specifically are `xenium_capture_window_um` (the real Xenium slide's 12x24mm
window, cropping cell-level output) and the single-cell `output_modalities`
selection itself -- Visium/VisiumHD aggregate multiple cells per
observation, Xenium (here) does not.

In [ ]:
sphere_r_um = 6000.0          # NOT overridden for cell -- manuscript default (0.16% 3D packing fraction)

config = {
    # ==== 3D tissue geometry and cell placement ====
    "sphere_R_um": sphere_r_um,
    "center": (0.0, 0.0, 0.0),                 # default
    "n_cells": 600_000,                        # manuscript scale
    "allow_cell_overlap": False,               # default
    "cell_radius_kwargs": dict(                # r_mean/r_sigma tuned; r_min/r_max scaled proportionally
        radius_dist="lognormal", r_mean=7.5, r_sigma=0.28, r_min=4.0, r_max=14.0,
    ),

    # ==== Spatial domains with irregular boundaries (manuscript-wide, all 4 pairings) ====
    "n_domains": 6,
    "core_frac": 0.55,
    "core_bump_amp": 0.25,
    "wedge_angle_amp_deg": 25.0,
    "noise_terms": 16,
    "noise_freq_range": (3.0, 6.0),
    "boundary_fuzz_width_deg": 6.0,
    "boundary_fuzz_flip_prob": 0.15,
    "core_fuzz_width_um": 0.05 * sphere_r_um,  # manuscript ratio: 300um / 6000um sphere radius
    "core_fuzz_flip_prob": 0.25,

    # ==== Cell type composition by domain ====
    "n_cell_types": 8,
    "domain_type_mix": None,   # filled in below (manuscript_domain_type_mix)

    # ==== Gene programs and per-cell gene expression ====
    "marker_genes_per_type": 80,               # default
    "noise_gene_frac": 0.10,                   # default
    "shared_marker_frac": 0.25,                # default
    "base_gene_lognormal": (-2.3, 0.7),        # TUNED: lowers avg. per-cell total counts vs. default (0.7, 0.7)
    "marker_foldchange": 3.5,                  # default
    "shared_marker_foldchange": 2.5,           # default
    "noise_scale": 1.3,                        # TUNED: manuscript "noisy" baseline (default 0.9)
    "cell_size_lognormal": (0.0, 0.35),        # default
    "domain_size_factors": None,               # default (no per-domain expression shift)
    "theta": 0.40,                             # TUNED: NB dispersion (real Xenium fits theta_hat ~0.15-0.16)
    "theta_jitter": 0.15,                      # TUNED: per-gene spread of theta around 0.40

    # ==== Transcript-level molecule simulation ====
    "inside_prob": 0.95,                       # default
    "assign_k": 8,                             # default

    # ==== Platform-specific capture windows ====
    "xenium_capture_window_um": (12000.0, 24000.0),  # real Xenium slide window (12x24mm) -- crops cell output
    "capture_window_um": (6500.0, 6500.0),           # unused for cell output, kept for completeness
    "capture_window_center_um": (0.0, 0.0),          # default
    "bin_size_um": 8.0,                              # unused for cell output
    "spot_spacing_um": 100.0,                        # unused for cell output
    "spot_radius_um": 27.5,                          # unused for cell output

    # ==== Slicing, batch effects, and unaligned coordinates ====
    "n_slices": 10,
    "slice_axes": ("Z",),
    "batch_sigma": 1.5,                        # TUNED: per-slice technical batch effect (see tutorial README caveats)
    "max_deg": 270.0,
    "max_shift": 0.5 * sphere_r_um,            # manuscript ratio: 3000um / 6000um sphere radius
    "base_seed_unaligned": 12345,              # default

    # ==== Simulation outputs ====
    "output_modalities": ("cell",),            # single-cell resolution -- the defining choice for "Xenium-like"
    "sparse_X": True,                          # default
    "seed": 2025,                              # default
}


In [ ]:
# Manuscript-baseline domain_type_mix (6 domains x 8 cell types). Every domain
# gets a distinct, non-uniform composition -- this is what Figure 2/3's
# domain_true and cell_type_true ground truth is built on. (An alternative
# "strong" mix with much sharper per-domain enrichment exists for dedicated
# domain-recovery test runs, but every Figure 2 count-distribution config uses
# this manuscript-baseline matrix.)
manuscript_domain_type_mix = np.array([
    [0.18, 0.18, 0.13, 0.12, 0.11, 0.10, 0.09, 0.09],
    [0.11, 0.12, 0.18, 0.18, 0.13, 0.10, 0.09, 0.09],
    [0.10, 0.11, 0.12, 0.13, 0.18, 0.18, 0.09, 0.09],
    [0.10, 0.10, 0.11, 0.12, 0.13, 0.13, 0.16, 0.15],
    [0.14, 0.13, 0.12, 0.11, 0.12, 0.13, 0.13, 0.12],
    [0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125],
])
config["domain_type_mix"] = manuscript_domain_type_mix


> **Runtime warning.** This notebook reproduces the manuscript's actual
> Figure 2 configuration at full scale (**600,000 cells**), not a reduced
> interactive-scale example like `higher_level_api_tutorial.ipynb` /
> `lower_level_api_tutorial.ipynb`. 600,000 cells at single-cell resolution with no spatial aggregation is the heaviest of the three tutorials in this set -- the manuscript run allocates up to 6 hours of SLURM walltime for this exact configuration. If you just want to see the
> API work end-to-end quickly, drop `n_cells` and `sphere_R_um` down (e.g.
> `n_cells=5000`, proportionally smaller `sphere_R_um`) -- every other
> parameter below is still the exact manuscript value.


## Generate

`simulate_3d_molecule_sphere_multires(**config)` builds the 3D tissue once,
then slices/captures/applies the batch effect for the requested modality.

In [ ]:
sim = ab.simulate_3d_molecule_sphere_multires(**config)

print("top-level keys:", list(sim.keys()))
adata = sim["adata_cell_sectioned"]["Z"]
adata


## Inspect the output

In [ ]:
print(adata)
print()
print("n_obs (observations):", adata.n_obs)
print("n_vars (genes):", adata.n_vars)
print("obs columns:", list(adata.obs.columns))
print("var columns:", list(adata.var.columns))


In [ ]:
summary = ab.describe(adata)
summary


### Empirical count dispersion

`generate_simulation_noisy.py` (the manuscript script this notebook mirrors)
fits an empirical per-gene NB dispersion from the pre-batch counts as a sanity
check against real platform data via
`sim_paper/code/count_distribution/count_distribution.py --compare-input`.
Same check here:

In [ ]:
def fit_empirical_theta(X):
    """Same method count_distribution.py uses to fit an empirical NB dispersion
    from a count matrix, for comparing against real platform data."""
    if hasattr(X, "multiply"):
        mean = np.asarray(X.mean(axis=0)).ravel()
        mean_sq = np.asarray(X.multiply(X).mean(axis=0)).ravel()
    else:
        mean = X.mean(axis=0)
        mean_sq = (X ** 2).mean(axis=0)
    var = mean_sq - mean ** 2
    overdispersed = var > mean
    theta_hat = mean[overdispersed] ** 2 / (var[overdispersed] - mean[overdispersed])
    theta_hat = theta_hat[np.isfinite(theta_hat) & (theta_hat > 0)]
    return float(np.median(theta_hat)) if theta_hat.size else float("nan")

raw_counts = adata.layers["counts_pre_batch"] if "counts_pre_batch" in adata.layers else adata.X
empirical_theta = fit_empirical_theta(raw_counts)
print(f"empirical theta_hat = {empirical_theta:.3f}")


## Plot

`ab.plot()` returns a Matplotlib figure. `coordinates="aligned"` shows the
canonical tissue coordinates (what `domain_true`/`cell_type_true` were
assigned against); `coordinates="unaligned"` shows the per-slice rigid
misalignment `max_deg`/`max_shift` introduce -- the same synthetic
registration-error ground truth used elsewhere in the manuscript.

In [ ]:
fig = ab.plot(adata, view="2d", coordinates="aligned", color="domain_true", point_size=3)
fig


In [ ]:
fig = ab.plot(adata, view="2d", coordinates="aligned", color="cell_type_true", point_size=3)
fig


In [ ]:
fig = ab.plot(adata, view="2d", coordinates="unaligned", color="slice_id", point_size=3)
fig


## Save

Same `ab.save(...)` helper as the other two tutorials -- writes an `AnnData`
to `.h5ad` and returns the path.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

output_path = ab.save(adata, output_dir / "xenium_cell_z.h5ad")
output_path
